# RAG Experiments (Staged Evaluation)

This notebook is dedicated to reproducible experiments.

Experiment setup: 2×3 Factorial Approach
| Configuration id | Embedding Model | Chunking Strategy |
| :--- | :--- | :--- |
| C1 | Nomic-embed-text | Fixed Size |
| C2 | Nomic-embed-text | Recursive |
| C3 | Nomic-embed-text | Semantic |
| C4 | BGE-M3 | Fixed Size |
| C5 | BGE-M3 | Recursive |
| C6 | BGE-M3 | Semantic |

Following measures are being made:
- Stage A: ingestion (`ingestion_seconds`)
- Stage B: retrieval-only (`retrieval_only_seconds`)
- Stage C: answer generation from retrieved docs (`generation_only_seconds`)
- Query-to-response metric: `query_to_response_seconds = Stage B + Stage C`

It also exports retrieved chunks for manual audit. RAGAS evaluation is commented out until there are benchmark questions and reference answers.

In [ ]:
import csv
import hashlib
import json
import os
import random
import time
import uuid
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, List, Literal

import chromadb
from IPython.display import Markdown, display
from langchain_classic.retrievers import MultiQueryRetriever
from langchain_chroma import Chroma
from langchain_community.chat_models import ChatOllama
from langchain_community.document_loaders import DirectoryLoader, UnstructuredPDFLoader
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_ollama import OllamaEmbeddings
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter
from langchain_experimental.text_splitter import SemanticChunker

/Users/alexandragalimurka/MSD/Semester5/Projektarbeit/rag_chatbot/venv_rag/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Core Components and Setup

In [ ]:
@dataclass
class ExperimentConfig:
    embedding_model: Literal["nomic-embed-text", "bge-m3"] = "nomic-embed-text"
    chunking_strategy: Literal["fixed", "recursive", "semantic"] = "semantic"
    chunk_params: Dict[str, Any] = None
    retriever_params: Dict[str, Any] = None

    llm_model: str = "llama3.2"
    llm_temperature: float = 0.1

    data_dir: str = "data_folder/"
    chroma_path: str = "chroma_database"
    collection_prefix: str = "rag_chatbot"

    logs_dir: str = "runs"
    random_seed: int = 42
    # Post-split cap for embedding APIs (Ollama nomic rejects oversized texts). None = auto.
    max_embedding_chars: int | None = None


def _defaults_if_missing(config: ExperimentConfig) -> ExperimentConfig:
    if config.chunk_params is None:
        config.chunk_params = {
            "chunk_size": 1000,
            "chunk_overlap": 200,
            "breakpoint_threshold_type": "percentile",
            "separator": "\n\n",
        }
    if config.retriever_params is None:
        config.retriever_params = {
            "k": 4,
            "fetch_k": 20,
            "lambda_mult": 0.6,
            "search_type": "mmr",
        }
    if config.max_embedding_chars is None:
        # SemanticChunker can exceed local embed context; nomic-embed-text via Ollama ~8k tokens.
        config.max_embedding_chars = (
            7000 if config.embedding_model.lower() == "nomic-embed-text" else 0
        )
    return config


def set_reproducibility(seed: int = 42) -> None:
    random.seed(seed)
    try:
        import numpy as np

        np.random.seed(seed)
    except Exception:
        pass


def _safe_collection_name(config: ExperimentConfig) -> str:
    # Separate collections by embedding/chunker to avoid embedding-dimension collisions.
    return f"{config.collection_prefix}_{config.embedding_model}_{config.chunking_strategy}".replace("-", "_")


def build_embeddings(config: ExperimentConfig):
    model = config.embedding_model.lower()

    if model == "nomic-embed-text":
        return OllamaEmbeddings(model="nomic-embed-text")

    if model == "bge-m3":
        try:
            from langchain_huggingface import HuggingFaceEmbeddings
        except Exception as e:
            raise RuntimeError(
                "BGE-M3 requires HuggingFace support. Install: pip install langchain-huggingface sentence-transformers"
            ) from e

        return HuggingFaceEmbeddings(model_name="BAAI/bge-m3")

    raise ValueError(f"Unsupported embedding model: {config.embedding_model}")


def build_chunker(config: ExperimentConfig, embeddings):
    strategy = config.chunking_strategy.lower()
    p = config.chunk_params

    if strategy == "fixed":
        return CharacterTextSplitter(
            chunk_size=int(p.get("chunk_size", 1000)),
            chunk_overlap=int(p.get("chunk_overlap", 200)),
            separator=p.get("separator", "\n\n"),
        )

    if strategy == "recursive":
        return RecursiveCharacterTextSplitter(
            chunk_size=int(p.get("chunk_size", 1000)),
            chunk_overlap=int(p.get("chunk_overlap", 200)),
        )

    if strategy == "semantic":
        return SemanticChunker(
            embeddings,
            breakpoint_threshold_type=p.get("breakpoint_threshold_type", "percentile"),
        )

    raise ValueError(f"Unsupported chunking strategy: {config.chunking_strategy}")


def load_documents(data_dir: str):
    loader = DirectoryLoader(
        data_dir,
        glob="**/*.pdf",
        loader_cls=UnstructuredPDFLoader,
        show_progress=True,
    )
    documents = loader.load()
    if not documents:
        raise RuntimeError(f"No PDF files found in '{data_dir}'.")

    # Add file path and hash to metadata - hash is used to identify the document in the database
    for doc in documents:
        source_path = doc.metadata.get("source", "")
        if source_path and os.path.exists(source_path):
            doc.metadata["file_path"] = source_path
            with open(source_path, "rb") as f:
                doc.metadata["file_hash"] = hashlib.md5(f.read()).hexdigest()

    return documents


def split_documents(documents, chunker):
    return chunker.split_documents(documents)


def build_vector_db(chunks, embeddings, config: ExperimentConfig):
    client = chromadb.PersistentClient(path=config.chroma_path)
    collection_name = _safe_collection_name(config)

    return Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        collection_name=collection_name,
        client=client,
    )


def build_retriever(vector_db, llm, config: ExperimentConfig):
    p = config.retriever_params
    base_retriever = vector_db.as_retriever(
        search_type=p.get("search_type", "mmr"),
        search_kwargs={
            "k": int(p.get("k", 4)),
            "fetch_k": int(p.get("fetch_k", 20)),
            "lambda_mult": float(p.get("lambda_mult", 0.6)),
        },
    )

    query_prompt = PromptTemplate(
        input_variables=["question"],
        template="""You are a query rephrasing assistant for vector search.
    Generate 2 alternative, semantically diverse versions of the user's question.
    Return only the rephrased questions, one per line.
    Original question: {question}""",
    )

    return MultiQueryRetriever.from_llm(base_retriever, llm, prompt=query_prompt)


def format_chunk_list(chunk_list):
    rows = []
    for chunk in chunk_list:
        src = os.path.basename(chunk.metadata.get("source", "Unknown"))
        rows.append(f"Document Source: {src}\nContent: {chunk.page_content}")
    return "\n\n---\n\n".join(rows)


def generate_answer_from_docs(question: str, docs, llm) -> str:
    template = """You are a helpful and accurate assistant. You answer in the same language as the question.

    Answer the question using ONLY the provided Context information below.
    If context is insufficient, explicitly say the information is not available.

    Context: {context}
    Question: {question}

    Answer:
    """
    prompt = ChatPromptTemplate.from_template(template)
    chain = prompt | llm | StrOutputParser()
    return chain.invoke({"context": format_chunk_list(docs), "question": question})


def _approx_token_count(text: str) -> int:
    return max(1, int(len(text) / 4))


def _chunk_stats(chunks):
    if not chunks:
        return {"chunk_count": 0, "avg_chunk_chars": 0.0, "avg_chunk_tokens_approx": 0.0}

    lengths = [len(c.page_content) for c in chunks]
    token_estimates = [_approx_token_count(c.page_content) for c in chunks]
    return {
        "chunk_count": len(chunks),
        "avg_chunk_chars": sum(lengths) / len(lengths),
        "avg_chunk_tokens_approx": sum(token_estimates) / len(token_estimates),
    }

# this value is stored in the run metrics (and CSV) so one can tell later which corpus a logged experiment used, without storing full paths in every row or re-reading all PDFs
def _dataset_fingerprint(documents):
    parts = []
    for d in documents:
        p = d.metadata.get("file_path", "")
        h = d.metadata.get("file_hash", "")
        parts.append(f"{p}:{h}")
    joined = "|".join(sorted(parts))
    return hashlib.md5(joined.encode("utf-8")).hexdigest()

## LOGGING

In [3]:
# logging single chunk for audit purposes
def export_retrieved_chunks_for_audit(
    run_id: str,
    question: str,
    docs,
    logs_dir: str,
    config_id: str = "",
    question_id: str = "",
):
    logs = Path(logs_dir)
    logs.mkdir(parents=True, exist_ok=True)

    payload = {
        "run_id": run_id,
        "config_id": config_id,
        "question_id": question_id,
        "question": question,
        "retrieved_k": len(docs),
        "chunks": [
            {
                "rank": i + 1,
                "source": d.metadata.get("source", ""),
                "file_path": d.metadata.get("file_path", ""),
                "file_hash": d.metadata.get("file_hash", ""),
                "content": d.page_content,
            }
            for i, d in enumerate(docs)
        ],
    }

    json_path = logs / f"{run_id}_retrieved_chunks.json"
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)

    csv_path = logs / "retrieved_chunks_audit.csv"
    rows = []
    for c in payload["chunks"]:
        rows.append(
            {
                "run_id": run_id,
                "config_id": config_id,
                "question_id": question_id,
                "question": question,
                "rank": c["rank"],
                "source": c["source"],
                "file_path": c["file_path"],
                "file_hash": c["file_hash"],
                "content": c["content"],
            }
        )

    default_fields = [
        "run_id",
        "config_id",
        "question_id",
        "question",
        "rank",
        "source",
        "file_path",
        "file_hash",
        "content",
    ]
    write_header = not csv_path.exists()
    with open(csv_path, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()) if rows else default_fields)
        if write_header:
            writer.writeheader()
        if rows:
            writer.writerows(rows)

    return str(json_path), str(csv_path)

# logging metrics from run_query_on_prepared_setup function into metrics.csv file
def log_experiment_row(config: ExperimentConfig, metrics: Dict[str, Any]):
    logs_dir = Path(config.logs_dir)
    logs_dir.mkdir(parents=True, exist_ok=True)

    run_id = metrics["run_id"]
    run_json_path = logs_dir / f"{run_id}.json"
    with open(run_json_path, "w", encoding="utf-8") as f:
        json.dump(metrics, f, ensure_ascii=False, indent=2)

    metrics_csv_path = logs_dir / "metrics.csv"
    row = {
        "run_id": run_id,
        "config_id": metrics.get("config_id", ""),
        "question_id": metrics.get("question_id", ""),
        "question": metrics.get("question", ""),
        "timestamp_utc": metrics["timestamp_utc"],
        "embedding_model": config.embedding_model,
        "chunking_strategy": config.chunking_strategy,
        "chroma_path": config.chroma_path,
        "ingestion_seconds": metrics["ingestion_seconds"],
        "retrieval_only_seconds": metrics["retrieval_only_seconds"],
        "generation_only_seconds": metrics["generation_only_seconds"],
        "query_to_response_seconds": metrics["query_to_response_seconds"],
        "ingestion_plus_query_seconds": metrics["ingestion_plus_query_seconds"],
        "chunk_count": metrics["chunk_count"],
        "avg_chunk_chars": metrics["avg_chunk_chars"],
        "avg_chunk_tokens_approx": metrics["avg_chunk_tokens_approx"],
        "k": config.retriever_params.get("k", 4),
        "fetch_k": config.retriever_params.get("fetch_k", 20),
        "dataset_fingerprint": metrics["dataset_fingerprint"],
    }

    write_header = not metrics_csv_path.exists()
    with open(metrics_csv_path, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(row.keys()))
        if write_header:
            writer.writeheader()
        writer.writerow(row)

    return str(run_json_path), str(metrics_csv_path)




# SETUP (Stage A) + QUERY RUNS (Stage B & C)

In [ ]:

# Stage A: ingestion is measured here - RUNS ONCE FOR CONFIG AND STAYS THE SAME FOR ALL QUESTIONS 
def prepare_config_setup(config: ExperimentConfig):
    """Build once per config: embeddings, chunks, vector DB, retriever, and ingestion metrics."""
    config = _defaults_if_missing(config)
    set_reproducibility(config.random_seed)

    t_a = time.perf_counter() # start time for ingestion
    embeddings = build_embeddings(config)
    chunker = build_chunker(config, embeddings)
    documents = load_documents(config.data_dir)
    chunks = split_documents(documents, chunker)
    vector_db = build_vector_db(chunks, embeddings, config)
    ingestion_seconds = time.perf_counter() - t_a

    retriever_llm = ChatOllama(model=config.llm_model, temperature=config.llm_temperature)
    retriever = build_retriever(vector_db, retriever_llm, config)
    answer_llm = ChatOllama(model=config.llm_model, temperature=config.llm_temperature)

    stats = _chunk_stats(chunks)
    setup = {
        "config": config,
        "documents": documents,
        "retriever": retriever,
        "answer_llm": answer_llm,
        "ingestion_seconds": round(ingestion_seconds, 4),
        "chunk_count": stats["chunk_count"],
        "avg_chunk_chars": round(stats["avg_chunk_chars"], 2),
        "avg_chunk_tokens_approx": round(stats["avg_chunk_tokens_approx"], 2),
        "dataset_fingerprint": _dataset_fingerprint(documents),
    }
    return setup

# Stage B and Stage C: retrieval and generation are measured here - RUNS FOR EACH QUESTION
def run_query_on_prepared_setup(
    setup: Dict[str, Any],
    question: str,
    config_id: str = "",
    question_id: str = "",
):
    """Run Stage B+C per question using one already-ingested config setup."""
    config = setup["config"]
    retriever = setup["retriever"]
    answer_llm = setup["answer_llm"]

    run_id = uuid.uuid4().hex[:12]
    ts = datetime.now(timezone.utc).isoformat()

    t_q = time.perf_counter()

    t_b = time.perf_counter()
    retrieved_docs = retriever.invoke(question)
    retrieval_only_seconds = time.perf_counter() - t_b

    t_c = time.perf_counter()
    answer = generate_answer_from_docs(question, retrieved_docs, answer_llm)
    generation_only_seconds = time.perf_counter() - t_c

    query_to_response_seconds = time.perf_counter() - t_q

    metrics = {
        "run_id": run_id,
        "config_id": config_id,
        "question_id": question_id,
        "question": question,
        "timestamp_utc": ts,
        "config": asdict(config),
        "ingestion_seconds": setup["ingestion_seconds"],
        "retrieval_only_seconds": round(retrieval_only_seconds, 4),
        "generation_only_seconds": round(generation_only_seconds, 4),
        "query_to_response_seconds": round(query_to_response_seconds, 4),
        "ingestion_plus_query_seconds": round(setup["ingestion_seconds"] + query_to_response_seconds, 4),
        "chunk_count": setup["chunk_count"],
        "avg_chunk_chars": setup["avg_chunk_chars"],
        "avg_chunk_tokens_approx": setup["avg_chunk_tokens_approx"],
        "dataset_fingerprint": setup["dataset_fingerprint"],
    }

    run_json_path, metrics_csv_path = log_experiment_row(config, metrics)
    chunks_json_path, chunks_csv_path = export_retrieved_chunks_for_audit(
        run_id=run_id,
        question=question,
        docs=retrieved_docs,
        logs_dir=config.logs_dir,
        config_id=config_id,
        question_id=question_id,
    )

    return {
        "answer": answer,
        "retrieved_docs": retrieved_docs,
        "metrics": metrics,
        "run_json": run_json_path,
        "metrics_csv": metrics_csv_path,
        "chunks_json": chunks_json_path,
        "chunks_csv": chunks_csv_path,
    }




In [5]:

# RAGAS: uncomment when you have benchmark Q&A and `pip install ragas datasets`
# def evaluate_with_ragas_scaffold(records: List[Dict[str, Any]]):
#     """Scaffold: records should include question, answer, contexts, and optional ground_truth."""
#     try:
#         from datasets import Dataset
#         from ragas import evaluate
#         from ragas.metrics import answer_relevancy, faithfulness
#     except Exception as e:
#         print("RAGAS not installed yet. Install with: pip install ragas datasets")
#         print(f"Import error: {e}")
#         return None
#
#     dataset = Dataset.from_list(records)
#     result = evaluate(dataset=dataset, metrics=[faithfulness, answer_relevancy])
#     return result

## Runner: Outer loop = configs (C1-C6), Inner loop = 12 queries

In [6]:

QUERIES_FILE = Path("12queries.json")


def load_queries_from_json(path: Path) -> List[Dict[str, str]]:
    payload = json.loads(path.read_text(encoding="utf-8"))
    if not isinstance(payload, list) or not payload:
        raise RuntimeError(f"No queries found in {path}")

    queries = []
    for i, row in enumerate(payload, start=1):
        if not isinstance(row, dict):
            raise RuntimeError(f"Invalid query row at index {i} in {path}")

        qid_raw = row.get("question_id")
        if not isinstance(qid_raw, int):
            raise RuntimeError(
                f"question_id must be int (row {i} in {path}); got {qid_raw!r}"
            )
        qid = str(qid_raw)
        question = str(row.get("question", "")).strip()
        if not question:
            raise RuntimeError(f"Missing question text for question_id={qid} in {path}")

        queries.append(
            {
                "question_id": qid,
                "question": question,
                "reference_answer": str(row.get("reference_answer", "")).strip(),
            }
        )

    return queries


benchmark_queries = load_queries_from_json(QUERIES_FILE)

BASE_CHROMA_DIR = Path("chroma_database")

CONFIGS = [
    {
        "config_id": "C1",
        "embedding_model": "nomic-embed-text",
        "chunking_strategy": "fixed",
        "chroma_path": str(BASE_CHROMA_DIR / "c1_nomic_fixed"),
    },
    {
        "config_id": "C2",
        "embedding_model": "nomic-embed-text",
        "chunking_strategy": "recursive",
        "chroma_path": str(BASE_CHROMA_DIR / "c2_nomic_recursive"),
    },
    {
        "config_id": "C3",
        "embedding_model": "nomic-embed-text",
        "chunking_strategy": "semantic",
        "chroma_path": str(BASE_CHROMA_DIR / "c3_nomic_semantic"),
    },
    {
        "config_id": "C4",
        "embedding_model": "bge-m3",
        "chunking_strategy": "fixed",
        "chroma_path": str(BASE_CHROMA_DIR / "c4_bge_fixed"),
    },
    {
        "config_id": "C5",
        "embedding_model": "bge-m3",
        "chunking_strategy": "recursive",
        "chroma_path": str(BASE_CHROMA_DIR / "c5_bge_recursive"),
    },
    {
        "config_id": "C6",
        "embedding_model": "bge-m3",
        "chunking_strategy": "semantic",
        "chroma_path": str(BASE_CHROMA_DIR / "c6_bge_semantic"),
    },
]

all_results = []

for cfg in CONFIGS:
    print(f"\n===== {cfg['config_id']} | {cfg['embedding_model']} + {cfg['chunking_strategy']} =====")

    config = ExperimentConfig(
        embedding_model=cfg["embedding_model"],
        chunking_strategy=cfg["chunking_strategy"],
        chunk_params={
            "chunk_size": 1000,
            "chunk_overlap": 200,
            "breakpoint_threshold_type": "percentile",
        },
        retriever_params={"k": 4, "fetch_k": 20, "lambda_mult": 0.6, "search_type": "mmr"},
        chroma_path=cfg["chroma_path"],
    )

    setup = prepare_config_setup(config)
    print(f"Ingestion done in {setup['ingestion_seconds']}s | chunks={setup['chunk_count']}")

    # Warm-up (not timed): avoid first-query latency spikes from model load/initialization.
    _ = setup["retriever"].invoke("Warm-up retrieval query.")
    _ = setup["answer_llm"].invoke("Warm-up generation. Reply with OK.")

    for q in benchmark_queries:
        result = run_query_on_prepared_setup(
            setup=setup,
            question=q["question"],
            config_id=cfg["config_id"],
            question_id=q["question_id"],
        )
        all_results.append(result)
        print(
            f"{q['question_id']}: retrieval={result['metrics']['retrieval_only_seconds']}s, "
            f"generation={result['metrics']['generation_only_seconds']}s, "
            f"q2r={result['metrics']['query_to_response_seconds']}s"
        )

print("\nBenchmark complete.")
print("Rows logged to:", all_results[-1]["metrics_csv"] if all_results else "(none)")
print("Retrieved chunks audit:", all_results[-1]["chunks_csv"] if all_results else "(none)")


===== C1 | nomic-embed-text + fixed =====


100%|██████████| 1/1 [00:44<00:00, 44.29s/it]
/var/folders/s9/zfjzp2lj0p3b4chlq2yxjs180000gp/T/ipykernel_11301/360421102.py:15: LangChainDeprecationWarning: The class `ChatOllama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import ChatOllama``.
  retriever_llm = ChatOllama(model=config.llm_model, temperature=config.llm_temperature)


Ingestion done in 48.4953s | chunks=77
1: retrieval=0.7466s, generation=3.8928s, q2r=4.6394s
2: retrieval=0.8195s, generation=3.9635s, q2r=4.783s
3: retrieval=1.0061s, generation=2.0529s, q2r=3.059s
4: retrieval=0.7525s, generation=2.4221s, q2r=3.1745s
5: retrieval=1.4973s, generation=3.819s, q2r=5.3163s
6: retrieval=1.0107s, generation=2.9749s, q2r=3.9856s
7: retrieval=1.4886s, generation=4.3269s, q2r=5.8154s
8: retrieval=1.2578s, generation=4.5633s, q2r=5.8211s
9: retrieval=0.7899s, generation=2.7806s, q2r=3.5704s
10: retrieval=1.7397s, generation=3.6817s, q2r=5.4214s
11: retrieval=0.9138s, generation=3.4745s, q2r=4.3884s
12: retrieval=0.6137s, generation=4.2881s, q2r=4.9017s

===== C2 | nomic-embed-text + recursive =====


100%|██████████| 1/1 [00:03<00:00,  3.16s/it]


Ingestion done in 5.5814s | chunks=79
1: retrieval=0.8644s, generation=3.0855s, q2r=3.9498s
2: retrieval=0.6623s, generation=3.9182s, q2r=4.5806s
3: retrieval=0.8559s, generation=2.9716s, q2r=3.8274s
4: retrieval=0.8194s, generation=3.0478s, q2r=3.8671s
5: retrieval=1.5134s, generation=4.063s, q2r=5.5764s
6: retrieval=1.0094s, generation=3.4931s, q2r=4.5025s
7: retrieval=1.3223s, generation=4.9144s, q2r=6.2366s
8: retrieval=1.1211s, generation=3.2365s, q2r=4.3576s
9: retrieval=0.8393s, generation=3.5865s, q2r=4.4258s
10: retrieval=1.5922s, generation=4.0706s, q2r=5.6628s
11: retrieval=0.9193s, generation=3.4508s, q2r=4.37s
12: retrieval=0.6205s, generation=3.8701s, q2r=4.4906s

===== C3 | nomic-embed-text + semantic =====


100%|██████████| 1/1 [00:03<00:00,  3.07s/it]


ResponseError: the input length exceeds the context length (status code: 400)